# SummitBridge Health Plan - Claims Overview Analysis

**Analysis Period**: January 1, 2024 – December 31, 2024
**Notebook**: 04_exploratory_analysis/01_claims_overview.ipynb
**Purpose**: Comprehensive claims landscape - service categories, LOBs, claim types, network status

In [ ]:
# Setup
import sys
sys.path.append('../../src')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data_loader import load_all
from src.utils import *

print_section("LOADING DATA")
data = load_all()
claims = data['claims']
enrollment = data['enrollment']
member_months = data['member_months']
providers = data['providers']
avoidable_ed = data['avoidable_ed']

print(f"Claims: {len(claims):,} rows")
print(f"Enrollment: {len(enrollment):,} rows")
print(f"Member Months: {len(member_months):,} rows")
print(f"Providers: {len(providers):,} rows")
print(f"Avoidable ED Reference: {len(avoidable_ed):,} rows")

In [ ]:
# Service Category Analysis
print_section("CLAIMS BY SERVICE CATEGORY")

cat_stats = claims.groupby('service_category').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean'),
    unique_members=('member_id', 'nunique')
).sort_values('total_allowed', ascending=False)

cat_stats['pct_claims'] = cat_stats['claim_count'] / cat_stats['claim_count'].sum() * 100
cat_stats['pct_allowed'] = cat_stats['total_allowed'] / cat_stats['total_allowed'].sum() * 100

print(cat_stats.to_string())
print(f"\nTotal Allowed: ${claims['allowed_amount'].sum():,.2f}")

In [ ]:
# Visualization: Spend by Service Category
fig = px.pie(
    cat_stats.reset_index(),
    values='total_allowed',
    names='service_category',
    title='Total Allowed Spend by Service Category (2024)',
    color_discrete_sequence=COLOR_SEQ
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

In [ ]:
# Claims by Line of Business
print_section("CLAIMS BY LINE OF BUSINESS")

lob_stats = claims.groupby('lob').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean'),
    unique_members=('member_id', 'nunique')
).sort_values('total_allowed', ascending=False)

lob_stats['pct_claims'] = lob_stats['claim_count'] / lob_stats['claim_count'].sum() * 100
lob_stats['pct_allowed'] = lob_stats['total_allowed'] / lob_stats['total_allowed'].sum() * 100

print(lob_stats.to_string())

In [ ]:
# Visualization: Spend by LOB
fig = px.bar(
    lob_stats.reset_index(),
    x='lob',
    y='total_allowed',
    title='Total Allowed Spend by Line of Business',
    labels={'total_allowed': 'Total Allowed ($)', 'lob': 'Line of Business'},
    color='lob',
    color_discrete_sequence=COLOR_SEQ,
    text_auto='.2s'
)
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# Claims by Claim Type
print_section("CLAIMS BY CLAIM TYPE")

type_stats = claims.groupby('claim_type').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean')
).sort_values('total_allowed', ascending=False)

print(type_stats.to_string())

In [ ]:
# In-Network vs Out-of-Network
print_section("IN-NETWORK vs OUT-OF-NETWORK")

net_stats = claims.groupby('in_network_flag').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean')
)

print(net_stats.to_string())
print(f"\nOON Rate: {net_stats.loc['N', 'claim_count'] / net_stats['claim_count'].sum() * 100:.1f}% of claims")
print(f"OON Allowed %: {net_stats.loc['N', 'total_allowed'] / net_stats['total_allowed'].sum() * 100:.1f}% of allowed")

In [ ]:
# Place of Service Codes
print_section("PLACE OF SERVICE CODES")

pos_stats = claims.groupby('place_of_service_code').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean')
)

pos_labels = {11: 'Office', 20: 'Urgent Care', 21: 'Inpatient Hospital', 22: 'Outpatient Hospital', 23: 'Emergency Room'}
pos_stats.index = pos_stats.index.map(pos_labels)
print(pos_stats.to_string())

In [ ]:
# Service Category x LOB Cross-tab
print_section("SERVICE CATEGORY x LOB")

cross = claims.groupby(['service_category', 'lob']).agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum')
).reset_index()

pivot_count = cross.pivot(index='service_category', columns='lob', values='claim_count').fillna(0)
pivot_allowed = cross.pivot(index='service_category', columns='lob', values='total_allowed').fillna(0)

print("Claim Count:")
print(pivot_count.to_string())
print("\nTotal Allowed:")
print(pivot_allowed.to_string())

In [ ]:
# Heatmap: Allowed by Service Category x LOB
fig = px.imshow(
    pivot_allowed,
    title='Allowed Amount by Service Category x LOB',
    color_continuous_scale='Blues',
    text_auto='.2s',
    aspect='auto'
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Top 20 Members by Allowed Amount
print_section("TOP 20 MEMBERS BY ALLOWED AMOUNT")

member_spend = claims.groupby('member_id').agg(
    total_allowed=('allowed_amount', 'sum'),
    claim_count=('claim_id', 'count')
).sort_values('total_allowed', ascending=False)

print(member_spend.head(20).to_string())

In [ ]:
# Summary Statistics
print_section("KEY SUMMARY STATISTICS")

total_allowed = claims['allowed_amount'].sum()
total_claims = len(claims)
unique_members = claims['member_id'].nunique()
avg_allowed_per_claim = claims['allowed_amount'].mean()
avg_allowed_per_member = total_allowed / unique_members

print(f"Total Allowed: ${total_allowed:,.2f}")
print(f"Total Claims: {total_claims:,}")
print(f"Unique Members: {unique_members:,}")
print(f"Avg Allowed per Claim: ${avg_allowed_per_claim:,.2f}")
print(f"Avg Allowed per Member: ${avg_allowed_per_member:,.2f}")
print(f"\nInpatient Avg: ${claims[claims['service_category']=='Inpatient']['allowed_amount'].mean():,.2f}")
print(f"ED Avg: ${claims[claims['service_category']=='ED']['allowed_amount'].mean():,.2f}")
print(f"Specialty Rx Avg: ${claims[claims['service_category']=='Specialty Rx']['allowed_amount'].mean():,.2f}")
print(f"PCP Avg: ${claims[claims['service_category']=='PCP']['allowed_amount'].mean():,.2f}")
print(f"Urgent Care Avg: ${claims[claims['service_category']=='Urgent Care']['allowed_amount'].mean():,.2f}")
print(f"Behavioral Health Avg: ${claims[claims['service_category']=='Behavioral Health']['allowed_amount'].mean():,.2f}")